# RTMA hourly EMC export

Pulls NWS Real-Time Mesoscale Analysis (`NOAA/NWS/RTMA`, hourly 2.5 km, 2011–present),
computes **per-pixel** RH, VPD, and EMC (so the nonlinear EMC(T, RH) is not biased by
`EMC(mean(T), mean(RH))`), then reduces to pyrome-mean hourly scalars and exports CSVs.

The downstream `fb_tools.weather.rtma` module consumes these CSVs to produce
FlamMap percentile-scenario FM (FM1/FM10/FM100 via hourly NFDRS78 time-lag with
Bradshaw 1984 precip saturation-stall; FM_herb/FM_woody via GSI).

**Runs on any pyrome subset.** Set `REGION` + `PYROME_IDS` in the config cell; the
fire-mask asset id, the reduction geometry, and the export filenames all derive from
those. Presets for the CO extent and the R4 forests (Uinta-Wasatch-Cache + Payette)
are in the config cell.

**Schema** (per export task — one per (year, month), one row per pyrome × hour):

```
pyrome_id, datetime_utc, tmp_f, rh_pct, emc_pct, vpd_pa, pcp_mm_hr
```

ERC stays GridMET-derived (FSPro/FSim daily contract). HRRR stays the wind source.
This pipeline only upgrades the dead-FM and (optionally) live-FM legs of the
per-pyrome FlamMap scenarios.

In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='cfri-ee')
print('GEE authenticated.')

GEE authenticated.


## Analysis region config

Set `REGION` (a short label used in the fire-mask asset id and output filenames)
and `PYROME_IDS` (any subset of `Pyromes_CONUS_20200206`). Everything downstream —
the fire mask, the per-image reduction, and the export filenames — derives from
these two variables, so the same notebook runs for CO, the R4 forests, or any
other pyrome subset.

Presets:
- **CO** analysis extent (`CLAUDE.md`): `[42, 43, 45, 46, 47, 52, 53, 56, 128]`
- **R4** forests — Uinta-Wasatch-Cache + Payette: `[14, 16, 44]`
  (pyrome 44 Wasatch-Uinta, 14 Idaho Batholith, 16 Blue Mountains)

In [2]:
# ── Analysis region config ───────────────────────────────────────────────
# Edit these two lines to target any pyrome subset.  REGION is a short label
# baked into the fire-mask asset id and the export filenames.
#   CO : [42, 43, 45, 46, 47, 52, 53, 56, 128]
#   R4 : [14, 16, 44]   # Uinta-Wasatch-Cache (44) + Payette (14, 16)
REGION     = 'InterWest'
PYROME_IDS = [42, 43, 45, 46, 47, 52, 53, 56, 128, 14, 16, 44]

pyromes = ee.FeatureCollection('projects/cfri-ee/assets/weather/Pyromes_CONUS_20200206')
region_pyromes = pyromes.filter(ee.Filter.inList('PYROME', PYROME_IDS))
print(f'{REGION} pyromes:', region_pyromes.size().getInfo())
region_pyromes.aggregate_array('PYROME').getInfo()

InterWest pyromes: 12


[14, 16, 42, 43, 44, 45, 46, 47, 52, 53, 56, 128]

## Fire-environment mask (MODIS burned area)

Restricts all subsequent RTMA spatial reductions to historically burned pixels.
Created once here; referenced as `fire_mask` inside `reduce_to_pyromes()`.

In [3]:
# ── Fire-environment mask from MODIS burned area (2001–2022) ──────────────
# Restricts RTMA spatial averaging to pixels that have historically burned,
# excluding high-elevation alpine / non-burnable areas that inflate pyrome-mean
# EMC and FM values.  Uses the permissive 'any sub-pixel burned' criterion to
# capture the full fire environment while removing clearly non-fire pixels.

# Capture MODIS native projection (500 m sinusoidal) ONCE.  ImageCollection
# reducers like .max() return a composite with NO fixed projection (EE assigns
# the default WGS84 / 1-degree grid), so we must re-anchor the composite to the
# native MODIS grid before reduceResolution — otherwise the 500 m → 2500 m
# aggregation degenerates into upsampling and yields blocky 1° burned/unburned
# cells, with whole pyromes reading as 0% fire-environment.
modis_proj = (
    ee.ImageCollection("MODIS/061/MCD64A1").first().select("BurnDate").projection()
)

modis_ba = (
    ee.ImageCollection("MODIS/061/MCD64A1")
      .filterDate("2001-01-01", "2026-01-01")
      .filterBounds(region_pyromes.geometry())
      .select("BurnDate")
      .map(lambda img: img.gt(0).unmask(0).byte())  # 1 = burned, 0 = not
)

# Any 500-m pixel burned at least once in the 22-year record.  Re-anchor the
# composite to the MODIS native 500 m projection so reduceResolution downsamples
# correctly (rather than treating the input as a 1° grid).
ever_burned_500m = modis_ba.max().setDefaultProjection(modis_proj)  # 0/1 at 500 m

# Aggregate to RTMA 2.5 km: cell is fire-environment if ≥1 sub-pixel burned
# (reduceResolution + reproject is the standard GEE downsampling pattern).
# 2500 / 500 = 5 → ~25 input pixels per output cell; maxPixels=256 leaves
# headroom for boundary partials.
REDUCE_SCALE = 2500  # RTMA native resolution (m); also used in reduce_to_pyromes

fire_mask = (
    ever_burned_500m
      .reduceResolution(reducer=ee.Reducer.max(), maxPixels=256)
      .reproject(crs="EPSG:4326", scale=REDUCE_SCALE)
      .selfMask()   # mask 0s → reduceRegions skips non-fire cells
)
print("Fire-environment mask created from MODIS MCD64A1 (2001–2022)")

Fire-environment mask created from MODIS MCD64A1 (2001–2022)


## Materialize the fire mask to an asset

The mask above is a *deferred* computation (MODIS `.max()` → `reduceResolution` →
`reproject`). Referencing it in `updateMask` re-derives that whole chain for every
hourly RTMA image — and `.reproject()` inside a mapped operation disables GEE's lazy
tile pyramid. Across ~5k images/year this is what caused the 12-hour export timeouts.

**Fix:** export the mask once to an Earth Engine asset, then load it back as a plain
(precomputed, pyramided) `ee.Image`. Run the export cell, wait for it to finish, then
run the load cell. After that, `fire_mask` carries no MODIS chain into the per-image graph.

In [4]:
# ── Export the fire mask to an Earth Engine asset (run ONCE per REGION) ────
# Run this cell once; wait for the task to finish (Code Editor Tasks tab or
# `mask_export_task.status()`), then run the load cell below.
# Asset id derives from REGION, so CO → co_fire_mask_modis_2500m and
# R4 → r4_fire_mask_modis_2500m — different regions never clobber each other.

FIRE_MASK_ASSET = f'projects/cfri-ee/assets/weather/{REGION.lower()}_fire_mask_modis_2500m'
FIRE_MASK_DESC  = f'{REGION.lower()}_fire_mask_modis_2500m'

mask_export_task = ee.batch.Export.image.toAsset(
    image=fire_mask,                          # selfMasked 0/1 fire-environment mask
    description=FIRE_MASK_DESC,
    assetId=FIRE_MASK_ASSET,
    region=region_pyromes.geometry().bounds(),
    scale=REDUCE_SCALE,                       # 2500 m
    crs='EPSG:4326',
    maxPixels= int(1e13),
    pyramidingPolicy={'.default': 'mode'},    # preserve burned/not at coarse levels
)
mask_export_task.start()
print('Submitted fire-mask asset export →', FIRE_MASK_ASSET)
print('Wait for completion before running the load cell:', mask_export_task.status()['state'])

Submitted fire-mask asset export → projects/cfri-ee/assets/weather/interwest_fire_mask_modis_2500m
Wait for completion before running the load cell: READY


In [5]:
# ── Load the materialized fire mask (run AFTER the asset export completes) ──
# Reassigns `fire_mask` to the stored asset so every downstream reduction
# references a precomputed raster instead of re-deriving the MODIS .max() →
# reduceResolution → reproject chain for each of the ~5k hourly images per year.
# This is the change that removes the export-timeout bottleneck.
fire_mask = ee.Image(FIRE_MASK_ASSET).selfMask()
print('Loaded fire mask from asset:', FIRE_MASK_ASSET)

Loaded fire mask from asset: projects/cfri-ee/assets/weather/interwest_fire_mask_modis_2500m


### Sanity check — mask coverage per pyrome

Run once manually before submitting exports. All pyromes should have ≥ 20% fire-environment pixel coverage.

In [6]:
# ── Sanity-check: fire-environment pixel coverage per pyrome ──────────────
# Run this once manually before the export loop to confirm the mask is
# non-trivial for all pyromes.  Expect ≥ 20% coverage in every pyrome.
# (A low-elevation desert-basin pyrome may fall below 20% — decide whether to
# keep it before exporting.)

mask_stats = (
    fire_mask.unmask(0)
      .reduceRegions(
          collection=region_pyromes,
          reducer=ee.Reducer.mean(),   # mean of 0/1 = fraction fire-environment
          scale=REDUCE_SCALE,
          tileScale=4,
      )
      .getInfo()
)
for feat in mask_stats["features"]:
    pid  = feat["properties"].get("PYROME")
    frac = feat["properties"].get("mean", 0) or 0
    flag = " ⚠ < 20%" if frac < 0.05 else ""
    print(f"  Pyrome {pid:>3d}: {frac*100:5.1f}% fire-environment pixels{flag}")

  Pyrome  14:  57.4% fire-environment pixels
  Pyrome  16:  38.2% fire-environment pixels
  Pyrome  42:   4.9% fire-environment pixels ⚠ < 20%
  Pyrome  43:  17.5% fire-environment pixels
  Pyrome  44:  29.6% fire-environment pixels
  Pyrome  45:  18.2% fire-environment pixels
  Pyrome  46:  20.4% fire-environment pixels
  Pyrome  47:  16.4% fire-environment pixels
  Pyrome  52:   6.7% fire-environment pixels
  Pyrome  53:  39.7% fire-environment pixels
  Pyrome  56:  12.6% fire-environment pixels
  Pyrome 128:   5.7% fire-environment pixels


## RTMA ImageCollection — inspect bands

RTMA on GEE exposes `TMP` (2-m temp, **°C**), `DPT` (dew point, **°C**), `SPFH`
(specific humidity, kg/kg), and `ACPC01` (hourly accumulated precip, kg/m² ≡ mm).
Note: the GEE catalog lists TMP and DPT in °C — **not Kelvin** — despite some
third-party examples treating them as K. Confirm with the range check below.

In [7]:
rtma = ee.ImageCollection('NOAA/NWS/RTMA')
sample_img = rtma.filterDate('2020-07-15', '2020-07-16').first()
print('Bands:', sample_img.bandNames().getInfo())
print('Sample date:', ee.Date(sample_img.get('system:time_start')).format('YYYY-MM-dd HH:mm').getInfo())

# Sanity-check TMP range over CONUS — expect ~-40 to +45 °C, NOT 230–320 K
tmp_stats = sample_img.select('TMP').reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=sample_img.geometry(),
    scale=10000,
    bestEffort=True,
).getInfo()
print('TMP min/max (should be °C, not K):', tmp_stats)

Bands: ['HGT', 'PRES', 'TMP', 'DPT', 'UGRD', 'VGRD', 'SPFH', 'WDIR', 'WIND', 'GUST', 'VIS', 'TCDC', 'ACPC01']
Sample date: 2020-07-15 00:00
TMP min/max (should be °C, not K): {'TMP_max': 47.6662422180176, 'TMP_min': 1.4237457275390852}


## Per-pixel transforms — RH, VPD, EMC

All computed on RTMA's native grid *before* the pyrome-mean reduction so the
nonlinear NFDRS EMC function isn't biased.

- RH from T, T_d via Magnus saturation vapor pressure ratio
- VPD = e_s(T) − e_s(T_d), in Pa
- EMC: three-regime piecewise NFDRS function (Cohen & Deeming 1985), see
  `fb_tools/weather/nfdrs.py:calc_emc` for the local equivalent.

In [8]:
TMP_BAND = 'TMP'      # 2-m temperature, °C (NOT Kelvin — GEE RTMA spec)
DPT_BAND = 'DPT'      # 2-m dew point temperature, °C (NOT Kelvin)
PCP_BAND = 'ACPC01'   # 1-hour accumulated precipitation, kg/m² ≡ mm

def add_derived_bands(img):
    """Append tmp_f, rh_pct, vpd_pa, emc_pct, pcp_mm_hr per-pixel bands."""
    # TMP and DPT are already in °C per the RTMA band specification.
    t_c  = img.select(TMP_BAND)
    td_c = img.select(DPT_BAND)

    # Magnus saturation vapor pressure (hPa)
    es_t  = t_c.multiply(17.67).divide(t_c.add(243.5)).exp().multiply(6.112)
    es_td = td_c.multiply(17.67).divide(td_c.add(243.5)).exp().multiply(6.112)
    rh    = es_td.divide(es_t).multiply(100).clamp(0, 100).rename('rh_pct')
    # VPD in Pa (1 hPa = 100 Pa)
    vpd_pa = es_t.subtract(es_td).multiply(100).max(0).rename('vpd_pa')

    tmp_f = t_c.multiply(1.8).add(32).rename('tmp_f')

    # EMC three-regime (Cohen & Deeming 1985): R<10, 10<=R<50, R>=50
    emc_low  = rh.multiply(0.281073).subtract(tmp_f.multiply(rh).multiply(0.000578)).add(0.03229)
    emc_mid  = rh.multiply(0.160107).subtract(tmp_f.multiply(0.014784)).add(2.22749)
    emc_high = (
        rh.pow(2).multiply(0.005565)
          .subtract(tmp_f.multiply(rh).multiply(0.00035))
          .subtract(rh.multiply(0.483199))
          .add(21.0606)
    )
    emc = emc_high.where(rh.lt(50), emc_mid).where(rh.lt(10), emc_low).rename('emc_pct')

    # Server-side conditional — bandNames().contains() avoids a client-side .getInfo()
    # call that would fail inside map().
    pcp = ee.Image(ee.Algorithms.If(
        img.bandNames().contains(PCP_BAND),
        img.select(PCP_BAND).rename('pcp_mm_hr'),
        ee.Image.constant(0).rename('pcp_mm_hr'),
    ))

    return img.addBands([tmp_f, rh, vpd_pa, emc, pcp])

## Per-image pyrome-mean reduction (fire-environment masked)

For each hourly image, apply the MODIS fire-environment mask then call
`reduceRegions` across all 9 CO pyrome geometries in a single batched operation.
Returns one feature per (pyrome, datetime_utc).

**Efficiency vs. prior implementation:**
- `reduceRegions` (one server op) replaces 9 serial `reduceRegion` calls per image.
- Fire mask eliminates alpine/non-burnable pixels (~high-elevation areas that inflate
  pyrome-mean EMC), improving representativeness of the spatial mean.
- `tileScale=4` prevents memory-limit errors on large pyrome geometries.

In [9]:
REDUCE_BANDS = ['tmp_f', 'rh_pct', 'emc_pct', 'vpd_pa', 'pcp_mm_hr']
# REDUCE_SCALE defined in the fire-mask cell above (2500 m)

def reduce_to_pyromes(img):
    """
    Apply fire-environment mask, compute derived bands, and reduce to
    pyrome-mean scalars in a single batched reduceRegions call.

    Efficiency notes vs. prior implementation:
    - reduceRegions (one server op) replaces N serial reduceRegion calls.
    - fire_mask eliminates non-burnable high-elevation pixels, reducing
      pixel count per pyrome and improving representativeness of the mean.
    - tileScale=4 avoids memory-limit errors on large pyrome geometries.
    """
    img_derived = (
        add_derived_bands(img)
          .select(REDUCE_BANDS)
          .updateMask(fire_mask)   # restrict to fire-environment pixels
    )

    # Timestamp computed once, server-side (avoids .getInfo() inside map)
    dt = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd HH:mm:ss")

    return (
        img_derived
          .reduceRegions(
              collection=region_pyromes,
              reducer=ee.Reducer.mean(),
              scale=REDUCE_SCALE,
              tileScale=4,
          )
          .map(lambda f: ee.Feature(None, {
              "pyrome_id":    f.get("PYROME"),
              "datetime_utc": dt,
              "tmp_f":        f.get("tmp_f"),
              "rh_pct":       f.get("rh_pct"),
              "emc_pct":      f.get("emc_pct"),
              "vpd_pa":       f.get("vpd_pa"),
              "pcp_mm_hr":    f.get("pcp_mm_hr"),
          }))
    )

## Quick test — single fire-season day

Sanity-check the reducer on one day before kicking off year-long exports.

In [10]:
test_day = (
    rtma
    .filterDate('2020-07-15', '2020-07-16')
    .filterBounds(region_pyromes.geometry())
)
print('Hourly images in test day:', test_day.size().getInfo())

test_fc = ee.FeatureCollection(test_day.map(reduce_to_pyromes).flatten())
print(f'Features (~24 hr × {len(PYROME_IDS)} pyromes):', test_fc.size().getInfo())

# Pull a small sample for inspection
test_fc.limit(24).getInfo()['features']

Hourly images in test day: 24
Features (~24 hr × 12 pyromes): 288


[{'type': 'Feature',
  'geometry': None,
  'id': '2020071500_0000000000000000000d',
  'properties': {'datetime_utc': '2020-07-15 00:00:00',
   'emc_pct': 5.98285266486197,
   'pyrome_id': 14,
   'rh_pct': 29.54697512988099,
   'tmp_f': 66.28509964310835,
   'vpd_pa': 1619.79418320101}},
 {'type': 'Feature',
  'geometry': None,
  'id': '2020071500_0000000000000000000f',
  'properties': {'datetime_utc': '2020-07-15 00:00:00',
   'emc_pct': 3.9811380213485306,
   'pyrome_id': 16,
   'rh_pct': 18.112883776332417,
   'tmp_f': 77.03398702723727,
   'vpd_pa': 2687.98408348702}},
 {'type': 'Feature',
  'geometry': None,
  'id': '2020071500_00000000000000000029',
  'properties': {'datetime_utc': '2020-07-15 00:00:00',
   'emc_pct': 3.3166996292036544,
   'pyrome_id': 42,
   'rh_pct': 14.797586303416285,
   'tmp_f': 86.13384937301909,
   'vpd_pa': 3698.994440073375}},
 {'type': 'Feature',
  'geometry': None,
  'id': '2020071500_0000000000000000002a',
  'properties': {'datetime_utc': '2020-07-15 

## Fire-season exports — chunked per month (2011–2025)

**One export task per (year, month)** over April–October — 15 yr × 7 mo = **105 tasks**.
A single per-year graph (~5k hourly images) timed out at 12 h; per-month tasks (~730
images each) stay well under the compute limit and give partial progress instead of
all-or-nothing. Each task is ~30 d × 24 h × N pyromes rows.

**Prerequisite:** the REGION fire-mask asset must be exported and loaded (cells above) —
that is what removes the per-image MODIS recompute that drove the timeouts.

**Robustness:** RTMA's `ACPC01` precip band is missing from some hours, so `submit_month`
selects bands per-image (`_keep_present`) rather than hard-selecting `ACPC01` — that hard
select is what failed ~6 months. `add_derived_bands` already treats a missing `ACPC01` as
0 mm precip, so those hours are handled correctly.

**Resume after partial failure:** `missing_months()` reads the EE task list and returns the
(year, month) pairs whose task is not `COMPLETED` (failures + never-submitted). Re-run only
those with `[submit_month(y, m) for (y, m) in missing_months()]` — no need to re-export the
months that already succeeded.

**File naming:** `rtma_hourly_{REGION}_pyromes_YYYY_MM.csv`. Same column schema. No loader
change needed: `fb_tools.weather.rtma.load_rtma_csv` accepts a directory, globs all
`*.csv`, then concatenates + sorts by `(pyrome_id, datetime_utc)` + dedupes — so point it
at the per-REGION download folder and the monthly files load as one frame. Keep each
REGION's CSVs in their own folder so different regions don't co-mingle.

Output: Google Drive folder `fb_tools_weather/`.

In [12]:
EXPORT_FOLDER = 'fb_tools_weather'
EXPORT_PREFIX = f'rtma_hourly_{REGION}_pyromes'   # e.g. rtma_hourly_CO_pyromes / rtma_hourly_InterWest_pyromes
YEARS  = range(2011, 2026)  # 2011-2025
MONTHS = range(4, 11)       # April (4) – October (10), fire season

_WANTED_BANDS = ['TMP', 'DPT', 'ACPC01']

def _keep_present(img):
    """Select only the wanted bands that THIS image actually carries.

    RTMA's precip band ACPC01 is absent from some analysis hours.  A hard
    .select(['TMP','DPT','ACPC01']) fails the whole task on the first such
    image ("Band pattern 'ACPC01' did not match any bands"), which is why a
    handful of months failed while the rest exported fine.  add_derived_bands()
    already treats a missing ACPC01 as 0 mm precip, so dropping it for those
    images is correct — we just must not hard-select it.
    """
    present = img.bandNames().filter(ee.Filter.inList('item', _WANTED_BANDS))
    return img.select(present)

def submit_month(year, month):
    """Submit one fire-season MONTH as its own export task.

    Per-month chunking keeps each export graph small enough to finish under the
    compute limit (one monolithic per-year graph timed out at 12 h).  Combined
    with the materialized fire-mask asset, each task reduces ~730 hourly images.
    """
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    season = (
        rtma.filterDate(start, end)
            .filterBounds(region_pyromes.geometry())
            .map(_keep_present)   # robust per-image band select (ACPC01 may be absent)
    )
    fc = ee.FeatureCollection(season.map(reduce_to_pyromes).flatten())

    name = f'{EXPORT_PREFIX}_{year}_{month:02d}'
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=name,
        folder=EXPORT_FOLDER,
        fileNamePrefix=name,
        fileFormat='CSV',
        selectors=['pyrome_id', 'datetime_utc', 'tmp_f', 'rh_pct', 'emc_pct', 'vpd_pa', 'pcp_mm_hr'],
    )
    task.start()
    return task

# ── Resume support — only re-run months that did NOT complete ─────────────
# Reads the Earth Engine task list (authoritative, works before you download
# anything) and returns the (year, month) pairs whose export task COMPLETED.
def completed_months(prefix=EXPORT_PREFIX):
    done = set()
    for t in ee.batch.Task.list():
        st = t.status()
        desc, state = st.get('description', ''), st.get('state')
        if state == 'COMPLETED' and desc.startswith(prefix + '_'):
            try:
                y, m = desc[len(prefix) + 1:].split('_')
                done.add((int(y), int(m)))
            except ValueError:
                pass
    return done

def missing_months(years=YEARS, months=MONTHS, prefix=EXPORT_PREFIX):
    """(year, month) tasks that are not COMPLETED — i.e. FAILED or never submitted."""
    done = completed_months(prefix)
    return [(y, m) for y in years for m in months if (y, m) not in done]

# Alternatively, derive the completed set from CSVs you've already downloaded
# locally (filename-based) instead of the EE task list:
#   import re, pathlib
#   def completed_months_local(folder, prefix=EXPORT_PREFIX):
#       pat = re.compile(rf'{re.escape(prefix)}_(\d{{4}})_(\d{{2}})\.csv$')
#       return {(int(y), int(m)) for f in pathlib.Path(folder).glob('*.csv')
#               for y, m in pat.findall(f.name)}

# --- workflow ---------------------------------------------------------------
# 1. See what still needs running (the ACPC01 failures + anything unsubmitted):
todo = missing_months()
print(f'{len(todo)} month-tasks not COMPLETED:', todo)

# 2. Re-submit ONLY those (skips everything that already exported):
tasks = [submit_month(y, m) for (y, m) in todo]
for t in tasks: print(t.status()['description'], t.status()['state'])

# --- first-time / full-batch options ---------------------------------------
# tasks = [submit_month(2020, 7)]                                 # single-month test
# tasks = [submit_month(2020, m) for m in MONTHS]                 # single-year test
# tasks = [submit_month(y, m) for y in YEARS for m in MONTHS]     # full batch (105 tasks)

72 month-tasks not COMPLETED: [(2011, 6), (2011, 7), (2012, 5), (2012, 10), (2013, 4), (2013, 8), (2013, 10), (2014, 4), (2014, 7), (2014, 8), (2014, 9), (2014, 10), (2015, 9), (2015, 10), (2017, 9), (2017, 10), (2018, 4), (2018, 5), (2018, 6), (2018, 7), (2018, 8), (2018, 9), (2018, 10), (2019, 4), (2019, 5), (2019, 6), (2019, 7), (2019, 8), (2019, 9), (2019, 10), (2020, 4), (2020, 5), (2020, 6), (2020, 7), (2020, 8), (2020, 9), (2020, 10), (2021, 4), (2021, 5), (2021, 6), (2021, 7), (2021, 8), (2021, 9), (2021, 10), (2022, 4), (2022, 5), (2022, 6), (2022, 7), (2022, 8), (2022, 9), (2022, 10), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2025, 4), (2025, 5), (2025, 6), (2025, 7), (2025, 8), (2025, 9), (2025, 10)]
rtma_hourly_InterWest_pyromes_2011_06 RUNNING
rtma_hourly_InterWest_pyromes_2011_07 RUNNING
rtma_hourly_InterWest_pyromes_2012_05 RUNNING
rtma_hourly_InterWest_pyro